# Alfvén eigenmode - verification

Use the **Python (FAITH labelmaker)** kernel, started under the wrapper:
`pixi run -e labelmaker fdp run jupyter lab`. Set `shot` below, drag a time
range on any panel, then press *Mark present* / *Mark absent*, *Verify* and
*Save*.

**No AE-annotated shot is in the corpus.** The 180 annotated shots span
170659-178879; the corpus starts at 185601 and only carries `co2` above
198279, so there is no overlap at all - every shot here is fetched live over
PTDATA instead. The first fetch of a shot moves ~240 MB (four CO2 chords at
1.667 MHz) and takes several minutes; after that it is cached under
`data/events/alfven_eigenmode/review/_cache/<shot>_co2.npz` and reopening the
same shot is instant. That cache is a fetch cache, not a review record: it
holds the raw record verbatim, costs about **240 MB per shot**, has no cap
and no eviction, and is safe to delete at any time - the next open of that
shot simply refetches it. `*.npz` is gitignored, so it cannot be committed by
accident.

**What you are looking for**: coherent narrowband activity inside 80-250 kHz
(the shaded band) that shows up on MORE THAN ONE chord pair. A feature on a
single pair only is more likely chord-specific noise than a real mode. TAEs
sit near the low end of the band; RSAEs sweep upward as the safety factor
evolves through the shot.

`tier` and `holdout` in `shots.csv` are curation calls set by hand; nothing
here derives them. *Save* records the reviewer and the date and does not
promote anything.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from scipy import signal as scipy_signal

MAX_TIME_BINS = 1000


def crosspower(time_ms, a, b, *, nperseg=2048, max_khz=300.0):
    """Log |S_a . conj(S_b)| for two chords, as (freq_khz, t_ms, z).

    The rate comes from the SPAN, never from a median of successive
    differences: a float32 time vector quantises its spacing at t ~ 3 s.
    """
    rate = (len(time_ms) - 1) / ((time_ms[-1] - time_ms[0]) / 1000.0)
    kwargs = {
        "fs": rate,
        "nperseg": nperseg,
        "noverlap": nperseg // 2,
        "mode": "complex",
    }
    freq, times_s, spec_a = scipy_signal.spectrogram(a, **kwargs)
    _, _, spec_b = scipy_signal.spectrogram(b, **kwargs)
    cross = spec_a * np.conj(spec_b)

    keep = freq <= max_khz * 1000.0
    freq_khz = freq[keep] / 1000.0
    magnitude = np.abs(cross[keep])
    t_ms = times_s * 1000.0 + time_ms[0]

    # A full-rate spectrogram of a 2 s window is ~3300 columns; three of
    # those as plotly heatmaps is tens of megabytes of JSON and a wedged
    # browser tab. Block-average the time axis instead, which keeps a chirp
    # over hundreds of milliseconds perfectly visible.
    #
    # The average is taken over the POWER and the log comes after it.
    # Averaging the log instead is a geometric mean, which is pulled down by
    # the quiet bins in a block and so suppresses exactly the short bursts
    # this panel exists to show.
    if magnitude.shape[1] > MAX_TIME_BINS:
        width = magnitude.shape[1] // MAX_TIME_BINS
        usable = (magnitude.shape[1] // width) * width
        magnitude = (
            magnitude[:, :usable]
            .reshape(len(freq_khz), -1, width)
            .mean(axis=2)
        )
        t_ms = t_ms[:usable].reshape(-1, width).mean(axis=1)
    return freq_khz, t_ms, np.log10(magnitude + 1e-30)

In [ ]:
from labeler.config import Paths
from labeler.events.verify import (
    CO2_CHORDS,
    NoDataError,
    Panel,
    corpus_signal,
    fdp_signal,
    review,
)

event = "alfven_eigenmode"
shot = 178642  # measured-good fetch case; has a label grid under format/shots
source = "format/shots"

# This window bounds the SPECTROGRAM's cost, not the fetch's - fdp_signal
# always fetches (and caches) the whole record regardless of t_range. The
# labels for 178642 run 0-1950 ms, present 350-1300 ms.
t_range = (0.0, 2000.0)

cache = (
    Paths.from_env().label_tables / event / "review" / "_cache" / f"{shot}_co2.npz"
)
try:
    co2 = corpus_signal(shot, "co2", t_range=t_range)
except NoDataError:
    # No AE-annotated shot is in the corpus - 170659-178879 against a corpus
    # starting at 185601 - so this is the normal path, not the exception.
    # ~240 MB and several minutes on the first fetch of a shot, then cached.
    co2 = fdp_signal(
        shot, list(CO2_CHORDS), via="ptdata", t_range=t_range, cache=cache
    )

AE_BAND = (80.0, 250.0)
# R0xV1, R0xV2, R0xV3 - CO2_CHORDS is ("DENR0UF", "DENV1UF", "DENV2UF", "DENV3UF")
chord_pairs = [(0, 1), (0, 2), (0, 3)]

panels = []
for r0, vn in chord_pairs:
    freq_khz, t_ms, power = crosspower(co2.x, co2.y[r0], co2.y[vn])
    panels.append(
        Panel(
            title=f"CO2 crosspower {CO2_CHORDS[r0]} x {CO2_CHORDS[vn]}",
            kind="heatmap",
            x=t_ms,
            y=freq_khz,
            z=power,
            ylabel="kHz",
            bands=[AE_BAND],
        )
    )

In [ ]:
session = review(event, shot, panels, source=source)
session

After pressing *Save*, check what was written:

```python
from labeler.events.verify import read_corrections, review_path
read_corrections(review_path(event, shot))
```
